# Imports


In [21]:
from __future__ import annotations

import math
import pickle
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split

# Constants


In [22]:
RANDOM_SEED = 42

In [23]:
DATA_DIR: Path = Path("/home/linkezio/Datasets/cifar-10-python/cifar-10-batches-py")

In [24]:
MODELS_DIR: Path = Path("/home/linkezio/Projects/Efficient-Polling-Based-Learning-Rate-Optimization-for-Neural-Networks/models")

In [25]:
EPOCHS = 400

# Configs


## Seeds


In [26]:
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Device


In [27]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

device: cuda


# Data


## Dataset Class


In [28]:
# Loads one CIFAR batch and returns images and labels.
def _load_cifar_batch(path: Path) -> tuple[np.ndarray, np.ndarray]:
    with path.open("rb") as f:
        obj = pickle.load(f, encoding="bytes")
    data = obj[b"data"]  # (N, 3072)
    labels = np.array(obj.get(b"labels") or obj.get(b"fine_labels"), dtype=np.int64)
    images = data.reshape(-1, 3, 32, 32)
    return images, labels


In [29]:
class CIFAR10Dataset(Dataset):
    def __init__(
        self,
        data_dir: Path,
        train: bool,
        mean: torch.Tensor | None = None,
        std: torch.Tensor | None = None,
    ):
        if train:
            batch_files = [data_dir / f"data_batch_{i}" for i in range(1, 6)]
        else:
            batch_files = [data_dir / "test_batch"]

        xs = []
        ys = []
        for p in batch_files:
            if not p.exists():
                raise FileNotFoundError(f"File not found: {p}")
            x, y = _load_cifar_batch(p)
            xs.append(x)
            ys.append(y)

        images = np.concatenate(xs, axis=0)
        labels = np.concatenate(ys, axis=0)

        self.images = torch.from_numpy(images).float()  # (N,3,32,32)
        self.labels = torch.from_numpy(labels).long()
        
        if (mean is None) ^ (std is None):
            raise ValueError("Pass `mean` and `std` together, or neither (data in [0, 1]).")
        self.mean = mean
        self.std = std

    # Returns the total number of dataset samples.
    def __len__(self) -> int:
        return int(self.labels.shape[0])

    # Returns one sample (x, y), with optional normalization.
    def __getitem__(self, idx: int):
        x = self.images[idx]
        if self.mean is not None:
            x = (x - self.mean) / self.std
        y = self.labels[idx]
        return x, y


## Calculate mean and std for normalize later


In [30]:

# Computes per-channel mean and standard deviation over the full dataset.
def compute_mean_std(dataset, batch_size=512):
    loader_mean = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    channel_sum = None
    n_pixels = 0
    for images, _ in loader_mean:
        b, c, h, w = images.shape
        if channel_sum is None:
            channel_sum = torch.zeros(c, dtype=torch.float64)
        channel_sum += images.double().sum(dim=(0, 2, 3))
        n_pixels += b * h * w

    mean = (channel_sum / n_pixels).float()

    loader_var = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    sum_sq = None
    for images, _ in loader_var:
        b, c, h, w = images.shape
        if sum_sq is None:
            sum_sq = torch.zeros(c, dtype=torch.float64)
        diff = images.double() - mean.view(1, c, 1, 1).double()
        sum_sq += (diff * diff).sum(dim=(0, 2, 3))

    var = (sum_sq / n_pixels).float()
    std = torch.sqrt(var)
    std = torch.clamp(std, min=1e-8)
    return mean, std


In [31]:
train_for_stats = CIFAR10Dataset(DATA_DIR, train=True)

cifar_mean, cifar_std = compute_mean_std(train_for_stats)

cifar_mean = cifar_mean.view(3, 1, 1)
cifar_std = cifar_std.view(3, 1, 1)

print("mean (R,G,B):", cifar_mean.squeeze().tolist())
print("std  (R,G,B):", cifar_std.squeeze().tolist())

/tmp/ipykernel_43916/3686208571.py:4: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  obj = pickle.load(f, encoding="bytes")


mean (R,G,B): [125.30691528320312, 122.95039367675781, 113.86538696289062]
std  (R,G,B): [62.993221282958984, 62.088706970214844, 66.70490264892578]


## Train, Validation, Test Split


In [32]:
class DataLoaderHyperparameters:
    batch_size: int = 128
    val_fraction: float = 0.1
    num_workers: int = 0 # Jupyter: use 0 (workers cannot resolve classes defined in __main__ during pickling).

data_loader_hyperparameters = DataLoaderHyperparameters()

In [33]:
full_train = CIFAR10Dataset(DATA_DIR, train=True, mean=cifar_mean, std=cifar_std)
test_ds = CIFAR10Dataset(DATA_DIR, train=False, mean=cifar_mean, std=cifar_std)

val_size = max(1, int(len(full_train) * data_loader_hyperparameters.val_fraction))
train_size = len(full_train) - val_size

train_ds, val_ds = random_split(
    full_train,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(RANDOM_SEED),
)

train_loader = DataLoader(
    train_ds,
    batch_size=data_loader_hyperparameters.batch_size,
    shuffle=True,
    num_workers=data_loader_hyperparameters.num_workers,
    pin_memory=(DEVICE == "cuda"),
)
val_loader = DataLoader(
    val_ds,
    batch_size=data_loader_hyperparameters.batch_size,
    shuffle=False,
    num_workers=data_loader_hyperparameters.num_workers,
    pin_memory=(DEVICE == "cuda"),
)
test_loader = DataLoader(
    test_ds,
    batch_size=data_loader_hyperparameters.batch_size,
    shuffle=False,
    num_workers=data_loader_hyperparameters.num_workers,
    pin_memory=(DEVICE == "cuda"),
)

len(train_ds), len(val_ds), len(test_ds)


/tmp/ipykernel_43916/3686208571.py:4: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  obj = pickle.load(f, encoding="bytes")


(45000, 5000, 10000)

# Model


## Hyperparameters


In [34]:
class ModelHyperparameters:
    batch_size: int = 128
    epochs: int = EPOCHS
    lr: float = 1e-4
    weight_decay: float = 5e-4
    # Jupyter: use 0 (workers cannot resolve classes defined in __main__ during pickling).
    num_workers: int = 0

model_hyperparameters = ModelHyperparameters()

## Model Class


In [35]:
class SimpleCIFAR10CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, 10),
        )

    # Runs the model forward pass.
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# Training


## Metric Functions


In [36]:
# Computes average batch accuracy.
def accuracy(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = logits.argmax(dim=1)
    return (preds == y).float().mean().item()


In [37]:
loss_fn = nn.CrossEntropyLoss()

## Epoch Functions


In [38]:
@torch.inference_mode()
# Evaluates the model for one epoch and returns average loss/accuracy.
def eval_epoch(model: nn.Module, loader: DataLoader, loss_fn: nn.Module) -> tuple[float, float]:
    model.eval()
    losses = []
    accs = []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        logits = model(x)
        losses.append(loss_fn(logits, y).item())
        accs.append(accuracy(logits, y))
    return float(np.mean(losses)), float(np.mean(accs))


# Trains the model for one epoch and returns average loss/accuracy.
def train_epoch(model: nn.Module, loader: DataLoader, optim: torch.optim.Optimizer, loss_fn: nn.Module) -> tuple[float, float]:
    model.train()
    losses = []
    accs = []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optim.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optim.step()

        losses.append(loss.item())
        accs.append(accuracy(logits.detach(), y))
    return float(np.mean(losses)), float(np.mean(accs))


In [39]:
model = SimpleCIFAR10CNN().to(DEVICE)
optim = torch.optim.Adam(
    model.parameters(),
    lr=model_hyperparameters.lr,
    weight_decay=model_hyperparameters.weight_decay,
)

In [ ]:
# Coordinates epoch training/validation and saves the best checkpoint.
def fit_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optim: torch.optim.Optimizer,
    loss_fn: nn.Module,
    epochs: int,
    model_path: Path,
) -> tuple[dict[str, list[float]], float, Path]:
    """Coordinates epoch training/validation and saves the best checkpoint."""
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }

    best_val_acc = -math.inf

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optim, loss_fn)
        va_loss, va_acc = eval_epoch(model, val_loader, loss_fn)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), model_path)

        print(
            f"epoch {epoch:02d}/{epochs} | "
            f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
            f"val loss {va_loss:.4f} acc {va_acc:.4f}"
        )

    print("best val acc:", best_val_acc, "saved:", str(model_path))
    return history, best_val_acc, model_path


history, best_val_acc, model_path = fit_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optim=optim,
    loss_fn=loss_fn,
    epochs=model_hyperparameters.epochs,
    model_path=MODELS_DIR / "cifar10_best.pt",
)


epoch 01/400 | train loss 1.9697 acc 0.2638 | val loss 1.7960 acc 0.3363
epoch 02/400 | train loss 1.6993 acc 0.3706 | val loss 1.6417 acc 0.3922
epoch 03/400 | train loss 1.5947 acc 0.4084 | val loss 1.5942 acc 0.4178


In [ ]:
history, best_val_acc, model_path = fit_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optim=optim,
    loss_fn=loss_fn,
    epochs=model_hyperparameters.epochs,
    model_path=MODELS_DIR / "cifar10_best.pt",
)

# Test


In [ ]:
model.load_state_dict(torch.load(MODELS_DIR / "cifar10_best.pt", map_location=DEVICE))

test_loss, test_acc = eval_epoch(model, test_loader, loss_fn)

print(f"test loss {test_loss:.4f} | test acc {test_acc:.4f}")